In [ ]:
import os
import time
import math
import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats

# -----------------------------
# USER INPUTS
# -----------------------------
shapefile_path = r"C:\Users\KyleSteen.AzureAD\Documents\AEP_Workspace\CONUS\CONUS_3_3.shp"

aep_raster = r"C:\Users\KyleSteen.AzureAD\Documents\AEP_Workspace\CONUS\CONUS_AEP.tif"

log_file = r"C:\Users\KyleSteen.AzureAD\Documents\AEP_Workspace\CONUS_AEP_log.txt"
output_csv = r"C:\Users\KyleSteen.AzureAD\Documents\AEP_Workspace\CONUS_AEP.csv"

chunk_size = 10000

# -----------------------------
# Logging
# -----------------------------
def log(msg):
    print(msg)
    with open(log_file, "a") as f:
        f.write(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}\n")

# -----------------------------
# Remove old CSV if exists
# -----------------------------
if os.path.exists(output_csv):
    os.remove(output_csv)
    log("Existing output CSV removed.")

# -----------------------------
# Load shapefile
# -----------------------------
log(f"Loading shapefile: {shapefile_path}")
gdf = gpd.read_file(shapefile_path)

# Ensure required fields exist
required_fields = ["ROW_ID", "Square_Met"]  # adjust if needed
for field in required_fields:
    if field not in gdf.columns:
        raise ValueError(f"Missing required field: {field}")

total_polygons = len(gdf)
log(f"Loaded {total_polygons} polygons.")

# -----------------------------
# Compute chunks
# -----------------------------
num_chunks = math.ceil(total_polygons / chunk_size)
log(f"Processing in {num_chunks} chunks of {chunk_size} polygons each.")

# -----------------------------
# Process chunks
# -----------------------------
for chunk_idx in range(num_chunks):
    start_idx = chunk_idx * chunk_size
    end_idx = min((chunk_idx + 1) * chunk_size, total_polygons)
    chunk_gdf = gdf.iloc[start_idx:end_idx].copy()

    log(f"Processing chunk {chunk_idx + 1}/{num_chunks} ({start_idx} to {end_idx - 1})...")
    start_time = time.time()

    # Zonal statistics (ALL TOUCHED ENABLED)
    stats = zonal_stats(
        chunk_gdf,
        aep_raster,
        stats=["mean"],
        all_touched=True,
        nodata=-9999
    )

    # Add AEP mean
    chunk_gdf["AEP_mean"] = [s["mean"] for s in stats]

    # Keep only desired columns
    output_df = chunk_gdf[["ROW_ID", "Square_Met", "AEP_mean"]]

    # Append to CSV
    if chunk_idx == 0:
        output_df.to_csv(output_csv, index=False, mode="w")
    else:
        output_df.to_csv(output_csv, index=False, mode="a", header=False)

    elapsed = time.time() - start_time
    log(f"Chunk {chunk_idx + 1} processed in {elapsed:.2f} sec.")

log("All chunks processed successfully.")
log(f"Final CSV saved to: {output_csv}")